# UdaciSense: Optimized Object Recognition - GPU Cloud Version

## Notebook 1: Baseline Performance (GPU Accelerated)

This is a cloud-optimized version of the baseline notebook designed to run on Vertex AI Workbench with GPU acceleration.

**Key Changes for Cloud:**
- Increased batch sizes for GPU efficiency
- Google Cloud Storage integration for model persistence
- Optimized for T4 GPU training
- Enhanced monitoring and checkpointing

Remember, the CTO has set specific requirements:
- The optimized model should be **70% smaller** than the baseline
- The optimized model should **cut inference time by 60%**
- The optimized model should **maintain accuracy within 5%** of the baseline

### Step 0. Cloud Setup

In [ ]:
# Cell 0: Cloud environment setup
import subprocess
import sys

# Install required packages if not available
try:
    import google.cloud.storage
except ImportError:
    !pip install google-cloud-storage

# Set up project variables - UPDATE THESE
PROJECT_ID = "second-brain-463904"  # Your GCP project ID
BUCKET_NAME = f"{PROJECT_ID}-udacity-models"  # Cloud storage bucket
EXPERIMENT_NAME = "baseline-mobilenet-gpu"

print(f"Project ID: {PROJECT_ID}")
print(f"Bucket: {BUCKET_NAME}")
print(f"Experiment: {EXPERIMENT_NAME}")

In [ ]:
# Cell 1: Create GCS bucket and setup cloud storage
from google.cloud import storage
import os

# Initialize storage client
client = storage.Client(project=PROJECT_ID)

# Create bucket if it doesn't exist
try:
    bucket = client.create_bucket(BUCKET_NAME)
    print(f"Created bucket: {BUCKET_NAME}")
except Exception as e:
    bucket = client.bucket(BUCKET_NAME)
    print(f"Using existing bucket: {BUCKET_NAME}")

# Helper function to upload files to GCS
def upload_to_gcs(local_path, gcs_path):
    """Upload a file to Google Cloud Storage"""
    blob = bucket.blob(gcs_path)
    blob.upload_from_filename(local_path)
    print(f"Uploaded {local_path} to gs://{BUCKET_NAME}/{gcs_path}")

# Helper function to download files from GCS
def download_from_gcs(gcs_path, local_path):
    """Download a file from Google Cloud Storage"""
    blob = bucket.blob(gcs_path)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    blob.download_to_filename(local_path)
    print(f"Downloaded gs://{BUCKET_NAME}/{gcs_path} to {local_path}")

### Step 1. Set up the environment

In [ ]:
# Cell 2: Setup autoreload for development
%load_ext autoreload
%autoreload 2

In [ ]:
# Cell 3: Setup project structure for cloud environment
import sys
import os
print("Current working directory:", os.getcwd())

# Create project structure if running in cloud
project_root = os.path.abspath('../')
src_path = os.path.join(project_root, 'src')

if not os.path.exists(src_path):
    print("Creating project structure...")
    os.makedirs(src_path, exist_ok=True)
    os.makedirs(os.path.join(src_path, 'utils'), exist_ok=True)
    
    # You'll need to upload your src/ directory to the instance
    print("⚠️ Please upload your 'src/' directory to this instance")
    print("   You can do this via the Jupyter interface or gcloud scp")
else:
    print("✅ Project structure found")

# Add to Python path
sys.path.insert(0, project_root)
print(f"Added to Python path: {project_root}")

In [ ]:
# Cell 4: Install project requirements
!pip install -q torch>=2.0.0 torchvision>=0.15.0 numpy matplotlib seaborn pandas scikit-learn pillow tqdm plotly

In [ ]:
# Cell 5: Import all required libraries and custom modules
import json
import matplotlib.pyplot as plt
import numpy as np
import random
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
import warnings
warnings.filterwarnings('ignore')

# Try to import custom modules
try:
    from src.utils import MAX_ALLOWED_ACCURACY_DROP, TARGET_INFERENCE_SPEEDUP, TARGET_MODEL_COMPRESSION
    from src.utils.data_loader import get_household_loaders, get_input_size, print_dataloader_stats, visualize_batch
    from src.utils.model import MobileNetV3_Household, load_model, print_model_summary, train_model
    from src.utils.evaluation import calculate_confusion_matrix, evaluate_model_metrics
    from src.utils.visualization import plot_confusion_matrix, plot_training_history, plot_weight_distribution
    print("✅ All custom modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please ensure your 'src/' directory is uploaded to this instance")

In [ ]:
# Cell 6: Check device availability and set compute device (optimized for cloud GPU)
# Check available devices
devices = ["cpu"]
if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    for i in range(num_devices):
        device_name = torch.cuda.get_device_name(i)
        memory_gb = torch.cuda.get_device_properties(i).total_memory / 1024**3
        devices.append(f"cuda:{i} ({device_name}, {memory_gb:.1f}GB)")
        
print(f"Available devices: {devices}")

# Set device to cuda, if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🚀 Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")
    torch.cuda.empty_cache()  # Clear any existing GPU memory
    print("GPU cache cleared")

In [ ]:
# Cell 7: Set random seed for reproducibility
def set_deterministic_mode(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    
    def seed_worker(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    
    return seed_worker

set_deterministic_mode(42)
g = torch.Generator()
g.manual_seed(42)

print("✅ Deterministic mode set with seed=42")

In [ ]:
# Cell 8: Create directories for model artifacts and results
model_type = "baseline_mobilenet_gpu"
models_dir = f"../models/{model_type}"
models_ckp_dir = f"{models_dir}/checkpoints"
results_dir = f"../results/{model_type}"

os.makedirs(models_ckp_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

print(f"Created directories:")
print(f"  Models: {models_dir}")
print(f"  Checkpoints: {models_ckp_dir}")
print(f"  Results: {results_dir}")

### Step 2. Load the dataset (GPU optimized)

In [ ]:
# Cell 10: Load household objects dataset with GPU-optimized settings
# Increase batch size for GPU efficiency (was 128, now 256)
# Increase num_workers for faster data loading
GPU_BATCH_SIZE = 256 if torch.cuda.is_available() else 128
NUM_WORKERS = 4 if torch.cuda.is_available() else 2

print(f"Loading dataset with batch_size={GPU_BATCH_SIZE}, num_workers={NUM_WORKERS}")

train_loader, test_loader = get_household_loaders(
    image_size="CIFAR", 
    batch_size=GPU_BATCH_SIZE, 
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()  # Pin memory for faster GPU transfer
)

# Get class names
class_names = train_loader.dataset.classes
print(f"\nDatasets have {len(class_names)} classes:")
for i in range(len(class_names)):
    print(f"  {i}: {class_names[i]}")

# Print dataset statistics
for dataset_type, data_loader in [('train', train_loader), ('test', test_loader)]:
    print(f"\n{dataset_type.upper()} SET:")
    print_dataloader_stats(data_loader, dataset_type)

# Visualize some examples (smaller batch for visualization)
print("\nExamples from training set:")
visualize_batch(train_loader, num_images=8)

### Step 3. Train the baseline model (GPU accelerated)

In [ ]:
# Cell 12: Initialize baseline MobileNetV3 model
model = MobileNetV3_Household().to(device)
print_model_summary(model)

print(f"\n🚀 Model moved to {device}")
if torch.cuda.is_available():
    print(f"GPU Memory after model loading: {torch.cuda.memory_allocated()/1024**3:.2f}GB")

In [ ]:
# Cell 13: Define training configuration (GPU optimized)
# Reduce epochs for faster training on GPU (was 50, now 25)
num_epochs = 25
criterion = nn.CrossEntropyLoss()

# Slightly higher learning rate for larger batch size
base_lr = 0.001 * (GPU_BATCH_SIZE / 128)  # Scale LR with batch size
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=base_lr,
    weight_decay=1e-4,
    betas=(0.9, 0.999)
)

# Scale max_lr with batch size too
max_lr = 0.005 * (GPU_BATCH_SIZE / 128)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=max_lr,
    steps_per_epoch=len(train_loader),
    epochs=num_epochs,
    pct_start=0.3,
    div_factor=25,
    final_div_factor=1000
)

training_config = {
    'num_epochs': num_epochs,
    'criterion': criterion,
    'optimizer': optimizer,
    'scheduler': scheduler,
    'patience': 7,  # Increased patience
    'device': device
}

print(f"Training configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Base LR: {base_lr:.6f}")
print(f"  Max LR: {max_lr:.6f}")
print(f"  Batch size: {GPU_BATCH_SIZE}")
print(f"  Device: {device}")

In [ ]:
# Cell 14: Train the baseline model with cloud storage backup
import time

checkpoint_path = f"{models_ckp_dir}/model.pth"
gcs_checkpoint_path = f"{EXPERIMENT_NAME}/model.pth"

print(f"🚀 Starting training on {device}")
print(f"Expected training time on T4 GPU: ~15-20 minutes")
start_time = time.time()

# Train model
training_stats, best_accuracy, best_epoch = train_model(
    model,
    train_loader,
    test_loader,
    training_config,
    checkpoint_path=checkpoint_path,
)

training_time = time.time() - start_time
print(f"\n✅ Training completed in {training_time/60:.1f} minutes")
print(f"Best accuracy: {best_accuracy:.4f} at epoch {best_epoch}")

# Save training statistics locally and to cloud
stats_path = f"{results_dir}/training_stats.json"
with open(stats_path, 'w') as f:
    json.dump(training_stats, f, indent=4)

# Backup to cloud storage
try:
    upload_to_gcs(checkpoint_path, gcs_checkpoint_path)
    upload_to_gcs(stats_path, f"{EXPERIMENT_NAME}/training_stats.json")
    print("✅ Model and stats backed up to Google Cloud Storage")
except Exception as e:
    print(f"⚠️ Cloud backup failed: {e}")

### Step 4. Evaluate the baseline model

In [ ]:
# Cell 16: Evaluate baseline model performance and generate visualizations
# Load the best model
model = load_model(checkpoint_path, device)

# Define evaluation variables
class_names = test_loader.dataset.classes
n_classes = len(class_names)
input_size = get_input_size("CIFAR")

print("🔍 Evaluating model performance...")
evaluation_start = time.time()

# Calculate and save model performance metrics
metrics_path = f"{results_dir}/metrics.json"
baseline_metrics = evaluate_model_metrics(
    model, test_loader, device, n_classes, class_names, input_size, 
    save_path=metrics_path
)

evaluation_time = time.time() - evaluation_start
print(f"✅ Evaluation completed in {evaluation_time:.1f} seconds")

# Display key metrics
print(f"\n📊 KEY METRICS:")
print(f"  Top-1 Accuracy: {baseline_metrics['accuracy']['top1_acc']:.2f}%")
print(f"  Top-5 Accuracy: {baseline_metrics['accuracy']['top5_acc']:.2f}%")
print(f"  Model Size: {baseline_metrics['size']['model_size_mb']:.2f} MB")
print(f"  Parameters: {baseline_metrics['size']['num_params']:,}")
if torch.cuda.is_available():
    print(f"  GPU Inference Time: {baseline_metrics['timing']['cuda']['avg_time_ms']:.2f} ms")
print(f"  CPU Inference Time: {baseline_metrics['timing']['cpu']['avg_time_ms']:.2f} ms")

In [ ]:
# Cell 17: Generate and save visualizations
print("📈 Generating visualizations...")

# Calculate and plot confusion matrix
confusion_matrix = calculate_confusion_matrix(model, test_loader, device, n_classes)
cm_path = f"{results_dir}/confusion_matrix.png"
plot_confusion_matrix(confusion_matrix, class_names, cm_path)

# Plot training history
history_path = f"{results_dir}/training_history.png"
plot_training_history(training_stats, history_path)

# Plot weight distribution
weights_path = f"{results_dir}/weight_distribution.png"
plot_weight_distribution(model, output_path=weights_path)

print("✅ All visualizations generated")

# Backup visualizations to cloud
try:
    for local_path, gcs_name in [(cm_path, "confusion_matrix.png"), 
                                 (history_path, "training_history.png"),
                                 (weights_path, "weight_distribution.png"),
                                 (metrics_path, "metrics.json")]:
        upload_to_gcs(local_path, f"{EXPERIMENT_NAME}/{gcs_name}")
    print("✅ Visualizations backed up to cloud storage")
except Exception as e:
    print(f"⚠️ Cloud backup failed: {e}")

### Step 5. Calculate optimization targets

In [ ]:
# Cell 19: Calculate optimization targets based on CTO requirements
try:
    # Calculate target metrics
    target_model_size = baseline_metrics['size']['model_size_mb'] * (1 - TARGET_MODEL_COMPRESSION)
    target_inference_time_cpu = baseline_metrics['timing']['cpu']['avg_time_ms'] * (1 - TARGET_INFERENCE_SPEEDUP)
    min_acceptable_accuracy = baseline_metrics['accuracy']['top1_acc'] * (1 - MAX_ALLOWED_ACCURACY_DROP)
    
    print("🎯 OPTIMIZATION TARGETS:")
    print(f"  Model Size: {baseline_metrics['size']['model_size_mb']:.2f} → {target_model_size:.2f} MB ({TARGET_MODEL_COMPRESSION*100}% reduction)")
    print(f"  CPU Inference: {baseline_metrics['timing']['cpu']['avg_time_ms']:.2f} → {target_inference_time_cpu:.2f} ms ({TARGET_INFERENCE_SPEEDUP*100}% reduction)")
    
    if torch.cuda.is_available():
        target_inference_time_gpu = baseline_metrics['timing']['cuda']['avg_time_ms'] * (1 - TARGET_INFERENCE_SPEEDUP)
        print(f"  GPU Inference: {baseline_metrics['timing']['cuda']['avg_time_ms']:.2f} → {target_inference_time_gpu:.2f} ms ({TARGET_INFERENCE_SPEEDUP*100}% reduction)")
    
    print(f"  Min Accuracy: {baseline_metrics['accuracy']['top1_acc']:.2f} → {min_acceptable_accuracy:.2f}% (within {MAX_ALLOWED_ACCURACY_DROP*100}% drop)")
    
    # Save targets to file
    targets = {
        'baseline_size_mb': baseline_metrics['size']['model_size_mb'],
        'target_size_mb': target_model_size,
        'baseline_accuracy': baseline_metrics['accuracy']['top1_acc'],
        'min_accuracy': min_acceptable_accuracy,
        'baseline_cpu_ms': baseline_metrics['timing']['cpu']['avg_time_ms'],
        'target_cpu_ms': target_inference_time_cpu,
        'training_time_minutes': training_time / 60
    }
    
    if torch.cuda.is_available():
        targets.update({
            'baseline_gpu_ms': baseline_metrics['timing']['cuda']['avg_time_ms'],
            'target_gpu_ms': target_inference_time_gpu
        })
    
    targets_path = f"{results_dir}/optimization_targets.json"
    with open(targets_path, 'w') as f:
        json.dump(targets, f, indent=4)
    
    print(f"\n📁 All results saved to: {results_dir}/")
    
except NameError:
    print("❌ Could not import optimization constants. Please check your imports.")

### Step 6. Performance Summary

In [ ]:
# Cell 20: Display comprehensive performance summary
print("\n" + "="*60)
print("🏁 BASELINE TRAINING COMPLETE - PERFORMANCE SUMMARY")
print("="*60)

print(f"\n⏱️  TRAINING PERFORMANCE:")
print(f"   Training Time: {training_time/60:.1f} minutes")
print(f"   Best Epoch: {best_epoch}/{num_epochs}")
print(f"   Final Accuracy: {best_accuracy:.2f}%")

if torch.cuda.is_available():
    speedup = 12*60 / (training_time/60)  # Assuming 12 hours on CPU
    print(f"   🚀 GPU Speedup: ~{speedup:.1f}x faster than CPU")

print(f"\n📊 MODEL METRICS:")
print(f"   Accuracy: {baseline_metrics['accuracy']['top1_acc']:.2f}%")
print(f"   Model Size: {baseline_metrics['size']['model_size_mb']:.2f} MB")
print(f"   Parameters: {baseline_metrics['size']['num_params']:,}")

print(f"\n💾 SAVED ARTIFACTS:")
print(f"   Model: {checkpoint_path}")
print(f"   Metrics: {metrics_path}")
print(f"   Training Stats: {stats_path}")
print(f"   Visualizations: {results_dir}/*.png")

print(f"\n☁️  CLOUD BACKUP:")
print(f"   Bucket: gs://{BUCKET_NAME}/{EXPERIMENT_NAME}/")

print(f"\n🎯 NEXT STEPS:")
print(f"   1. Review the optimization analysis below")
print(f"   2. Implement compression techniques in notebook 02_compression.ipynb")
print(f"   3. Target: {target_model_size:.1f}MB size, {min_acceptable_accuracy:.1f}% accuracy")

print("="*60)

### Step 7. Optimization Analysis (Enhanced for GPU Results)

**Based on GPU-accelerated training results:**

In [ ]:
# Cell 21: Enhanced analysis with GPU performance data
gpu_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_available else "N/A"

analysis = f"""
# 🔬 Enhanced Optimization Analysis - GPU Training Results

## Training Environment
- **GPU**: {gpu_name if gpu_available else 'CPU Only'}
- **Training Time**: {training_time/60:.1f} minutes (vs ~12 hours on CPU)
- **Batch Size**: {GPU_BATCH_SIZE} (optimized for GPU)
- **Epochs**: {num_epochs} (reduced due to GPU efficiency)

## Baseline Performance Achieved
- **Accuracy**: {baseline_metrics['accuracy']['top1_acc']:.1f}% top-1, {baseline_metrics['accuracy']['top5_acc']:.1f}% top-5
- **Model Size**: {baseline_metrics['size']['model_size_mb']:.2f} MB ({baseline_metrics['size']['num_params']:,} parameters)
- **CPU Inference**: {baseline_metrics['timing']['cpu']['avg_time_ms']:.1f} ms"""

if gpu_available:
    analysis += f"""
- **GPU Inference**: {baseline_metrics['timing']['cuda']['avg_time_ms']:.1f} ms"""

analysis += f"""

## Optimization Strategy (Revised)

### 1. **Post-Training Quantization** (Primary)
**Why it's optimal for our GPU-trained model:**
- MobileNetV3 architecture is quantization-friendly
- Can achieve 4x size reduction (FP32 → INT8)
- GPU training ensures robust weight distributions
- Expected: ~1.5MB final size, <2% accuracy loss

### 2. **Structured Pruning** (Secondary)
**Channel-level pruning for mobile deployment:**
- Remove entire channels/filters for real speedup
- Focus on intermediate layers (avoid first/last layers)
- 20-30% parameter reduction achievable
- Complements quantization well

### 3. **Knowledge Distillation** (Fallback)
**If accuracy recovery needed:**
- Use this model as teacher
- Train smaller student or recover compressed model
- Can regain 2-3% accuracy points

## Recommended Pipeline
1. **Apply INT8 quantization** → Target: ~1.5MB, {baseline_metrics['accuracy']['top1_acc']-1:.1f}% accuracy
2. **If not meeting speed targets**: Add 20% structured pruning
3. **If accuracy drops too much**: Apply knowledge distillation

## Success Criteria
- ✅ Size: {baseline_metrics['size']['model_size_mb']:.1f}MB → {target_model_size:.1f}MB ({TARGET_MODEL_COMPRESSION*100}% reduction)
- ✅ Speed: {baseline_metrics['timing']['cpu']['avg_time_ms']:.1f}ms → {target_inference_time_cpu:.1f}ms ({TARGET_INFERENCE_SPEEDUP*100}% improvement)
- ✅ Accuracy: Keep above {min_acceptable_accuracy:.1f}% (within {MAX_ALLOWED_ACCURACY_DROP*100}% drop)

**The GPU training has provided a strong baseline - quantization alone should meet most targets!**
"""

print(analysis)

> 🚀 **Next Step:** 
> 
> **Your baseline model is now trained with GPU acceleration!**
> 
> Training time reduced from ~12 hours (CPU) to ~15-20 minutes (GPU T4)
> 
> Download your results or proceed to notebook `02_compression.ipynb` to implement the optimization techniques.